# B3 Gate15 X17 Prompt6 Copy-to-Root

Copy the Prompt5-validated candidate ZIP to `/kaggle/working/submission.zip` for manual Kaggle UI submission.

Safety scope:
- Copy only; no adapter generation.
- No ZIP regeneration.
- No SFT/training.
- No Kaggle Submit.
- Requires Prompt5 `validation_passed=true` and `safe_to_copy_to_root=true`.


In [ ]:
from pathlib import Path

EXPERIMENT_NAME = "B3_GATE15_X17_ASYMMETRIC_INPROJ_SPLIT"
BASE_DIR = Path("/kaggle/working/experiments/b3_gate15_x17")
DIAG_DIR = BASE_DIR / "diagnostics"
SOURCE_ZIP = BASE_DIR / "submission.zip"
ROOT_ZIP = Path("/kaggle/working/submission.zip")
DIAGNOSTIC_SUMMARY = DIAG_DIR / "diagnostic_summary.json"
COPY_CHECK_JSON = DIAG_DIR / "copy_to_root_check.json"
COPY_REPORT_MD = DIAG_DIR / "copy_to_root_report.md"

print("Prompt6 copy-to-root")
print("source:", SOURCE_ZIP)
print("root:", ROOT_ZIP)


In [ ]:
import hashlib
import json
import shutil
import traceback
import zipfile
from datetime import datetime, UTC
from pathlib import Path


def now_utc() -> str:
    return datetime.now(UTC).isoformat().replace("+00:00", "Z")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def list_zip_entries(path: Path) -> list[dict]:
    with zipfile.ZipFile(path, "r") as zf:
        return [
            {
                "name": info.filename,
                "file_size": info.file_size,
                "compress_size": info.compress_size,
                "crc": info.CRC,
            }
            for info in zf.infolist()
        ]


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def append_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(text)


def make_stop_report(reason: str, details: dict) -> None:
    DIAG_DIR.mkdir(parents=True, exist_ok=True)
    payload = {
        "status": "STOPPED",
        "reason": reason,
        "created_at": now_utc(),
        "experiment_name": EXPERIMENT_NAME,
        "details": details,
        "kaggle_submit_performed": False,
        "training_performed": False,
        "adapter_regenerated": False,
        "zip_regenerated": False,
    }
    write_json(DIAG_DIR / "STOP_REPORT_PROMPT6.json", payload)
    report = "\n".join([
        "# STOP_REPORT_PROMPT6",
        "",
        f"- status: STOPPED",
        f"- reason: {reason}",
        f"- created_at: {payload['created_at']}",
        "- kaggle_submit_performed: false",
        "- training_performed: false",
        "- adapter_regenerated: false",
        "- zip_regenerated: false",
        "",
        "## Details",
        "",
        "```json",
        json.dumps(details, ensure_ascii=False, indent=2),
        "```",
        "",
    ])
    (DIAG_DIR / "STOP_REPORT_PROMPT6.md").write_text(report, encoding="utf-8")


def fail(reason: str, details: dict):
    make_stop_report(reason, details)
    raise RuntimeError(f"STOP Prompt6: {reason}: {details}")

print("Prompt6 utilities loaded.")


In [ ]:
DIAG_DIR.mkdir(parents=True, exist_ok=True)

if not SOURCE_ZIP.exists():
    fail("source_zip_missing", {"source_zip": str(SOURCE_ZIP)})
if not DIAG_DIR.exists():
    fail("diagnostics_dir_missing", {"diag_dir": str(DIAG_DIR)})
if not DIAGNOSTIC_SUMMARY.exists():
    fail("diagnostic_summary_missing", {"diagnostic_summary": str(DIAGNOSTIC_SUMMARY)})

summary = json.loads(DIAGNOSTIC_SUMMARY.read_text(encoding="utf-8"))
required_summary_checks = {
    "validation_passed": summary.get("validation_passed") is True,
    "safe_to_copy_to_root": summary.get("safe_to_copy_to_root") is True,
    "kaggle_submit_not_performed": summary.get("kaggle_submit_not_performed") is True,
    "copy_to_root_not_performed": summary.get("copy_to_root_not_performed") is True,
}
failed_summary_checks = [k for k, ok in required_summary_checks.items() if not ok]
if failed_summary_checks:
    fail("diagnostic_summary_not_safe_for_copy", {"failed_summary_checks": failed_summary_checks, "summary": summary})

source_entries = list_zip_entries(SOURCE_ZIP)
source_entry_names = sorted(row["name"] for row in source_entries)
expected_entries = ["adapter_config.json", "adapter_model.safetensors"]
if source_entry_names != expected_entries:
    fail("source_zip_entries_unexpected", {"entries": source_entries, "entry_names": source_entry_names})

source_zip_size = SOURCE_ZIP.stat().st_size
source_zip_sha256 = sha256_file(SOURCE_ZIP)
existing_root_before = None
if ROOT_ZIP.exists():
    existing_root_before = {
        "path": str(ROOT_ZIP),
        "size_bytes": ROOT_ZIP.stat().st_size,
        "sha256": sha256_file(ROOT_ZIP),
        "entries": list_zip_entries(ROOT_ZIP),
    }

print("Prompt6 precheck passed")
print("source sha256:", source_zip_sha256)
print("source size:", source_zip_size)
print("source entries:", source_entry_names)
if existing_root_before:
    print("Existing root zip will be overwritten after recording metadata.")


In [ ]:
shutil.copy2(SOURCE_ZIP, ROOT_ZIP)

if not ROOT_ZIP.exists():
    fail("root_zip_missing_after_copy", {"root_zip": str(ROOT_ZIP)})

root_zip_size = ROOT_ZIP.stat().st_size
root_zip_sha256 = sha256_file(ROOT_ZIP)
root_entries = list_zip_entries(ROOT_ZIP)
root_entry_names = sorted(row["name"] for row in root_entries)
sha256_match = root_zip_sha256 == source_zip_sha256
size_match = root_zip_size == source_zip_size
expected_exact_entries_present = root_entry_names == expected_entries
unexpected_entries = sorted(set(root_entry_names) - set(expected_entries))

if not sha256_match:
    fail("root_zip_sha256_mismatch", {"source_sha256": source_zip_sha256, "root_sha256": root_zip_sha256})
if not size_match:
    fail("root_zip_size_mismatch", {"source_size": source_zip_size, "root_size": root_zip_size})
if not expected_exact_entries_present:
    fail("root_zip_entries_unexpected", {"entries": root_entries, "entry_names": root_entry_names})
if Path("/kaggle/working/adapter_model.safetensors").exists():
    fail("root_adapter_model_pollution", {"path": "/kaggle/working/adapter_model.safetensors"})
if Path("/kaggle/working/adapter_config.json").exists():
    fail("root_adapter_config_pollution", {"path": "/kaggle/working/adapter_config.json"})

copy_check = {
    "status": "PASS",
    "copied_at": now_utc(),
    "source_zip_path": str(SOURCE_ZIP),
    "root_zip_path": str(ROOT_ZIP),
    "source_zip_size": source_zip_size,
    "root_zip_size": root_zip_size,
    "source_zip_sha256": source_zip_sha256,
    "root_zip_sha256": root_zip_sha256,
    "sha256_match": sha256_match,
    "size_match": size_match,
    "zip_entries": root_entries,
    "expected_exact_entries_present": expected_exact_entries_present,
    "unexpected_entries": unexpected_entries,
    "existing_root_before": existing_root_before,
    "kaggle_submit_performed": False,
    "training_performed": False,
    "adapter_regenerated": False,
    "zip_regenerated": False,
    "safe_for_manual_kaggle_submit": True,
}
write_json(COPY_CHECK_JSON, copy_check)

zip_lines = "\n".join(f"- {row['name']} / {row['file_size']} bytes" for row in root_entries)
report = f"""# Prompt6 copy-to-root report

## Status

- status: PASS
- copied_at: {copy_check['copied_at']}
- Kaggle Submit performed: false
- SFT/training performed: false
- adapter regenerated: false
- ZIP regenerated: false

## Source ZIP

- path: `{SOURCE_ZIP}`
- size: {source_zip_size}
- sha256: `{source_zip_sha256}`

## Root ZIP

- path: `{ROOT_ZIP}`
- size: {root_zip_size}
- sha256: `{root_zip_sha256}`

## Checks

- sha256_match: {sha256_match}
- size_match: {size_match}
- expected_exact_entries_present: {expected_exact_entries_present}
- unexpected_entries: {unexpected_entries}

## ZIP entries

{zip_lines}

## Manual submit readiness

- safe_for_manual_kaggle_submit: true
- Submit must be performed manually by the user; this notebook did not call any Kaggle submit command.
"""
COPY_REPORT_MD.write_text(report, encoding="utf-8")

append_text(DIAG_DIR / "run_commands.md", f"""
# Prompt6 copy-to-root

- copied_at: {copy_check['copied_at']}
- source_zip: {SOURCE_ZIP}
- root_zip: {ROOT_ZIP}
- source_sha256: {source_zip_sha256}
- root_sha256: {root_zip_sha256}
- sha256_match: {sha256_match}
- size_match: {size_match}
- kaggle_submit_performed: false
- training_performed: false
- adapter_regenerated: false
- zip_regenerated: false
""")

print("Prompt6 PASS")
print(json.dumps(copy_check, ensure_ascii=False, indent=2))


In [ ]:
copy_check = json.loads(COPY_CHECK_JSON.read_text(encoding="utf-8"))
print("=== Prompt6 final summary ===")
print("Prompt6 status:", copy_check["status"])
print("source zip:", copy_check["source_zip_path"], copy_check["source_zip_sha256"], copy_check["source_zip_size"])
print("root zip:", copy_check["root_zip_path"], copy_check["root_zip_sha256"], copy_check["root_zip_size"])
print("sha256_match:", copy_check["sha256_match"])
print("size_match:", copy_check["size_match"])
print("ZIP entries:", [row["name"] for row in copy_check["zip_entries"]])
print("Kaggle Submit: not performed")
print("Manual Kaggle UI submit is now possible if the user chooses to submit this candidate.")
